# Notebook 7: SSD Deployment on the Raspberry Pi

This notebook runs **on the Raspberry Pi** (unlike Notebook 6, which runs
in Google Colab). It assumes you already ran Notebook 6 end to end and
copied the resulting `car_detection.onnx` into `models/` on both the
laptop and the Pi, per Notebook 6's closing instructions.

**If you haven't done that yet**, this notebook still works - every cell
below is written to detect that the model file is missing and explain
clearly what to do, rather than crashing. You can run this notebook now
to see how the pipeline is *supposed* to fit together, then re-run it for
real once `car_detection.onnx` exists.


## 1. The pipeline, conceptually

Every piece below already exists as tested code in `src/`:

```
Camera            src/vision/camera.py            capture_frame()
   |
   v
Resize/preprocess  src/vision/object_detection.py  _preprocess()
   |                (private - called internally by detect_cars())
   v
SSD inference       src/vision/object_detection.py  (onnxruntime session.run())
   |
   v
Raw detections       src/vision/object_detection.py  _parse_ssd_output()
   |
   v
Confidence filter    src/vision/object_detection.py  detect_cars()
   |                 (keeps only CAR_CLASS_ID detections above threshold)
   v
Bounding box(es)      this notebook                   drawn with cv2.rectangle
```

This notebook doesn't reimplement any of that pipeline - it **loads and
exercises the real `src/vision/object_detection.py`** exactly as
`ai_drive.py` does, plus one extra diagnostic step (Section 3) that looks
directly at the ONNX file's real input/output shapes, independent of
whatever `object_detection.py` assumes about them.


## 2. Load the model

### Explanation

Same `sys.path` trick every notebook in this project uses to import from
`src/` without it being an installed package. `CAR_DETECTION_MODEL_PATH` is
checked explicitly, up front, before anything tries to use it - so this
notebook can say clearly "the file isn't there yet" instead of letting a
later cell fail with a confusing error.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '../src')

from config import CAR_DETECTION_MODEL_PATH, CAR_CLASS_ID, CAR_DETECTION_CONFIDENCE_THRESHOLD
from vision.object_detection import get_car_detector, detect_cars
from vision.camera import get_camera, capture_frame, cleanup

model_path = Path(CAR_DETECTION_MODEL_PATH)
model_exists = model_path.exists()

if model_exists:
    print(f"Found model file at {model_path} ({model_path.stat().st_size:,} bytes).")
else:
    print(f"NO MODEL FILE FOUND at {model_path}.")
    print()
    print("Run Notebook 6 (in Google Colab) first, then copy the resulting")
    print("car_detection.onnx here, e.g. from your laptop:")
    print()
    print("  scp /home/salem/AI_Car_workshop/raspberry_pi_robot/models/car_detection.onnx \\")
    print("      admin@192.168.0.130:~/raspberry_pi_robot/models/car_detection.onnx")
    print()
    print("This notebook will still run below - get_car_detector() and detect_cars()")
    print("are both written to handle a missing model file safely (no crash, 'no")
    print("detection' every frame) - but Sections 3-6 will mostly just demonstrate")
    print("that graceful-skip behavior rather than showing a real detection.")


### Expected output

If Notebook 6 hasn't been run yet (the expected state the first time you
open this notebook): `NO MODEL FILE FOUND at .../models/car_detection.onnx`
plus the copy instructions above - this is expected, not an error.

Once the real model has been copied over: `Found model file at
.../models/car_detection.onnx (N,NNN,NNN bytes).`


## 3. Live model inspection

### Explanation

This loads `car_detection.onnx` directly with `onnxruntime.InferenceSession`
- separately from `get_car_detector()`, purely for this teaching/diagnostic
purpose - and prints its **real** input/output names, shapes, and dtypes.
This is exactly what Notebook 6's Section 18 did in Colab right after
exporting the model, so you can see the real format directly on the Pi
too, independent of any assumption `object_detection.py` makes about it.

> **If `detect_cars()` below doesn't work or produces nonsense:** compare
> this printout against `src/vision/object_detection.py`'s `_preprocess()`
> and `_parse_ssd_output()` - those were written as best-effort
> **PLACEHOLDERS** before a real model existed (see that file's module
> docstring), and likely need updating to match what's printed here.


In [ ]:
import onnxruntime as ort

if not model_exists:
    print("Skipping model inspection - no model file yet (see Section 2).")
else:
    inspect_session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])

    print("=" * 70)
    print("ONNX MODEL INPUT/OUTPUT SPEC (loaded directly, not assumed)")
    print("=" * 70)
    print("Inputs:")
    for inp in inspect_session.get_inputs():
        print(f"  name={inp.name!r}  shape={inp.shape}  dtype={inp.type}")
    print("Outputs:")
    for out in inspect_session.get_outputs():
        print(f"  name={out.name!r}  shape={out.shape}  dtype={out.type}")
    print("=" * 70)
    print()
    print("Compare this against src/vision/object_detection.py's assumptions:")
    print(f"  - _MODEL_INPUT_SIZE (placeholder): (300, 300)")
    print(f"  - CAR_CLASS_ID (placeholder, from config.py): {CAR_CLASS_ID}")
    print("  - expected output shape (placeholder): [1, 1, N, 7] "
          "(image_id, class_id, confidence, x1, y1, x2, y2)")


### Expected output

With no model file yet: `Skipping model inspection - no model file yet
(see Section 2).`

With a real model: a printed list of input(s) and output(s) with their
real names/shapes/dtypes, followed by a reminder of what
`object_detection.py` currently *assumes* those are. If the real shapes
differ from the placeholder assumptions (very possible - see Notebook 6
Section 18's own uncertainty about export format), that mismatch is the
diagnostic signal this section exists to surface.


## 4. Capture a real frame and run `detect_cars()`

### Explanation

This uses the actual project pipeline exactly as `ai_drive.py` calls it:
`get_car_detector()` (returns `None` safely if the model is missing/broken)
and `detect_cars(detector, frame)` (never raises, returns a clear "no
detection" result if `detector` is `None`). If a car is detected, its
box(es) are drawn on the frame with `cv2.rectangle`/`cv2.putText` and
displayed inline with `IPython.display.Image`, the same technique
Notebook 5 used for the camera feed (`cv2.imshow()` doesn't work in this
headless OpenCV install).


In [ ]:
import cv2
from IPython.display import Image, display

camera = get_camera()
detector = get_car_detector()
print(f"Camera ready. detector = {detector!r} "
      f"({'model loaded' if detector is not None else 'None - see Section 2'})")


### Expected output / Physical result

`Camera ready. detector = <onnxruntime.InferenceSession object at ...>
(model loaded)` once the real model exists, or `detector = None (None -
see Section 2)` if it doesn't yet - either way, no crash, matching
`object_detection.py`'s documented "safe with no model" behavior.

### Physical result

The camera's LED/activity indicator turns on (capture started); nothing
else moves yet.


In [ ]:
frame = capture_frame(camera)
result = detect_cars(detector, frame)

print(f"car_detected: {result['car_detected']}")
print(f"confidence:   {result['confidence']}")
print(f"raw detections this frame: {len(result['detections'])}")

frame_annotated = frame.copy()
h, w = frame.shape[:2]

if result["car_detected"]:
    # Redraw straight from the raw parsed detections list, applying the
    # same class/confidence filter detect_cars() applies internally (it
    # doesn't expose per-box results directly, only the single best match).
    for det in result["detections"]:
        if det["class_id"] != CAR_CLASS_ID or det["confidence"] <= CAR_DETECTION_CONFIDENCE_THRESHOLD:
            continue
        x1, y1, x2, y2 = det["bbox"]
        # bbox is documented (placeholder assumption) as normalized [0, 1]
        # of the model's input - see object_detection.py's module
        # docstring. Scale to this frame's real pixel size to draw it.
        pt1 = (int(x1 * w), int(y1 * h))
        pt2 = (int(x2 * w), int(y2 * h))
        cv2.rectangle(frame_annotated, pt1, pt2, (0, 255, 0), 2)
        label = f"car {det['confidence']:.2f}"
        cv2.putText(frame_annotated, label, (pt1[0], max(pt1[1] - 8, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    print("Drew bounding box(es) on the frame below.")
else:
    print("No car detected this frame - showing the raw frame with no boxes.")

ok, buf = cv2.imencode(".jpg", frame_annotated)
display(Image(data=buf.tobytes()))


### Expected output

With no model (or no car in frame): `car_detected: False`,
`confidence: None`, `raw detections this frame: 0`, and the plain
unannotated camera frame displayed below.

With a real model and a real car in frame: `car_detected: True`, a
confidence value, and the frame displayed with a green box and confidence
label drawn on it. **If the box looks wildly wrong (way off the actual
car, or the whole frame, or a tiny corner)** - that's the exact symptom
Section 3's note warns about: `_preprocess()`/`_parse_ssd_output()`
probably need updating to match the real model's spec printed in
Section 3.

### Physical result

None - this is a read-only capture, no motors/LEDs involved.


## 5. Confidence filtering: what changing the threshold does

### Explanation

`detect_cars()` takes `confidence_threshold` as a parameter (defaulting to
`config.py`'s `CAR_DETECTION_CONFIDENCE_THRESHOLD = 0.80`, the user's
requested value). Lower it and more/weaker detections start counting as
"a car"; raise it and only very confident detections do. This re-runs
detection on the **same already-captured frame** at a few different
thresholds so the effect is directly comparable, not confounded by the
scene changing between calls.


In [ ]:
for threshold in (0.3, 0.5, 0.8, 0.95):
    r = detect_cars(detector, frame, confidence_threshold=threshold)
    print(f"threshold={threshold:<5} car_detected={r['car_detected']!s:<6} "
          f"confidence={r['confidence']}")


### Expected output

Four lines, one per threshold. With no model: `car_detected=False` at
every threshold (nothing to filter). With a real model and a car in frame:
lower thresholds are at least as likely to report `car_detected=True` as
higher ones for the same frame (never the other way around) - if a
detection's confidence is, say, `0.62`, it will show as detected at
`threshold=0.3` and `threshold=0.5` but not at `threshold=0.8` or `0.95`.


## 6. Measure real performance - don't assume it

### Explanation

**Do not assume this is "real-time" - here is the actually measured
number.** This times `N` real, back-to-back `detect_cars()` calls on
freshly captured frames at whatever resolution `camera.py`'s
`get_camera()` is currently using (`DEFAULT_RESOLUTION`, 640x480 as of
this writing), and reports the average inference time in milliseconds and
the resulting frames-per-second. If the model file doesn't exist yet, this
is skipped with a clear message instead of reporting a meaningless number.


In [ ]:
import time

N_TRIALS = 25

if detector is None:
    print(f"Skipping performance measurement - no model loaded (detector is None).")
    print("Once car_detection.onnx exists and this cell is re-run, it will time "
          f"{N_TRIALS} real detect_cars() calls and report actual ms/FPS.")
else:
    timings_s = []
    for _ in range(N_TRIALS):
        perf_frame = capture_frame(camera)
        start = time.perf_counter()
        detect_cars(detector, perf_frame)
        timings_s.append(time.perf_counter() - start)

    avg_ms = (sum(timings_s) / len(timings_s)) * 1000
    fps = 1000 / avg_ms if avg_ms > 0 else float("inf")

    print(f"Measured over {N_TRIALS} real detect_cars() calls at {frame.shape[1]}x{frame.shape[0]}:")
    print(f"  average inference time: {avg_ms:.1f} ms")
    print(f"  resulting FPS:          {fps:.1f}")
    print(f"  min/max:                {min(timings_s)*1000:.1f} ms / {max(timings_s)*1000:.1f} ms")


### Expected output

With no model: a clear skip message, no fabricated numbers.

With a real model: `average inference time: NN.N ms`, `resulting FPS: N.N`,
and a min/max range. This is CPU-only inference on a Pi 4 - do not assume
it matches whatever FPS Colab's GPU achieved in Notebook 6; that number is
irrelevant here. `AI_DRIVE_LOOP_HZ` in `config.py` targets ~10Hz for the
*whole* detect-decide-act loop (color detection + optional car detection +
motor/LED decision), not car detection alone - compare this section's
measured FPS against that target to see how much headroom (or lack of it)
car detection alone leaves for everything else in that loop.


## 7. Cleanup

### Explanation

Release the camera, same "close what you opened" pattern as every
previous vision/hardware notebook.


In [ ]:
cleanup(camera)
print("Camera released.")


### Expected output

`Camera released.` printed, no error.

### Physical result

Camera activity indicator turns off.


## Recap

- This notebook runs **on the Raspberry Pi**, not in Colab - it deploys
  and tests the model Notebook 6 produced, using the real, already-tested
  `src/vision/object_detection.py` and `src/vision/camera.py` exactly as
  `ai_drive.py` calls them. Neither of those files was modified by this
  notebook.
- **Section 2** checks for the model file explicitly and explains what to
  do if it's missing, rather than letting a later cell fail confusingly -
  the rest of the notebook still runs safely either way, since
  `get_car_detector()`/`detect_cars()` are already written to degrade to
  "no detection" with no model present.
- **Section 3**'s live model inspection is the key diagnostic tool if
  detections come back wrong: it prints the ONNX file's *real*
  input/output spec directly from `onnxruntime`, to compare against
  `object_detection.py`'s placeholder `_preprocess()`/`_parse_ssd_output()`
  assumptions.
- **Section 6** measured actual inference time/FPS on real Pi hardware -
  never assume Colab's GPU numbers (or any theoretical number) transfer
  to CPU inference on the Pi.


## Exercises

**1. Change the confidence threshold and observe the false positive/false
negative trade-off.** Using Section 5's loop as a starting point, try a
range of thresholds on several different frames (car in frame, no car in
frame, car partially out of frame) and note where you start seeing false
positives (threshold too low) versus missed real cars (threshold too
high).

**2. Measure FPS at a different camera resolution.**
`get_camera()` accepts a `resolution` tuple - try
`get_camera(resolution=(320, 240))` and re-run Section 6. Does inference
get meaningfully faster at a lower resolution, or is the bottleneck
elsewhere (e.g. `_preprocess()` already resizes to a fixed model input
size regardless of the camera's capture resolution)?

**3. Count multiple car detections, if the model returns more than one.**
`result["detections"]` contains *all* raw parsed detections, not just the
single best car match `result["confidence"]` reports. Filter it for
`class_id == CAR_CLASS_ID` and count how many distinct cars (if any) show
up above threshold in one frame.

**4. Discuss (markdown only, no code needed): what would change to detect
a second object class?**
`CAR_CLASS_ID` in `config.py` is a single hardcoded class index, and
`detect_cars()`'s public return shape (`car_detected`/`confidence`) is
car-specific. Sketch out, in a markdown cell, what you'd need to add or
generalize (e.g. a list of target class ids instead of one, a
per-class-name result dict instead of a single boolean) to detect, say,
"car" *and* "person" at once - you don't need to implement it.
